# Hyperparameter Optimization

In [2]:
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score
from catboost import CatBoostClassifier
from skopt.space import Integer, Real

import sys 
from datetime import datetime
import shutil
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load / Reload Selection Utility Functions

In [3]:
from utils2 import optimization as hpo

----

## Read Config File

In [4]:
config_path = Path(r'experiments\binary')
config_filename =  "optimization_config_final.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict


{'experiment': {'summary': 'binary classification - hyperparameter optimization (final experiment)',
  'classification_type': 'binary',
  'stage': 'hyperparameter_optimization',
  'tag': 'final',
  'verbosity': 0,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'name': 'catboost'},
 'param_space': {'iterations': {'min': 100, 'max': 500},
  'depth': {'min': 4, 'max': 10},
  'learning_rate': {'min': 0.01, 'max': 0.1},
  'l2_leaf_reg': {'min': 1, 'max': 9}},
 'optimization': {'scoring': 'roc_auc',
  'k_splits_outer': 4,
  'n_repeats_outer': 10,
  'k_splits_inner': 3,
  'n_iter': 30},
 'evaluation': {'confidence': 0.95},
 'final_training': {'k_splits_inner': 3, 'n_iter': 30}}

#### Set output directory

In [5]:
outputdir = config_path /  config.experiment.tag / config.experiment.stage
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\final\hyperparameter_optimization


#### Copy config file to output directory

In [6]:
source = config_path / config_filename
destination = outputdir / config_filename
shutil.copy(source, destination)

WindowsPath('experiments/binary/final/hyperparameter_optimization/optimization_config_final.yml')

## Data Loading

In [12]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

## Optuna Bayes Search Optimization

In [5]:
config.param_space

namespace(iterations=namespace(min=100, max=500),
          depth=namespace(min=4, max=10),
          learning_rate=namespace(min=0.01, max=0.1),
          l2_leaf_reg=namespace(min=1, max=9))

In [6]:
hpo.model_class[config.model.name]

catboost.core.CatBoostClassifier

In [7]:
config.optimization

namespace(scoring='roc_auc',
          k_splits_outer=4,
          n_repeats_outer=10,
          k_splits_inner=3,
          n_iter=30)

In [8]:
def param_space_fn(trial):
    return  {
        "iterations": trial.suggest_int(
            "iterations", 
            config.param_space.iterations.min, 
            config.param_space.iterations.max),
        "depth": trial.suggest_int(
            "depth", 
            config.param_space.depth.min, 
            config.param_space.depth.max),
        "learning_rate": trial.suggest_float(
            "learning_rate", 
            config.param_space.learning_rate.min, 
            config.param_space.learning_rate.max, 
            log=True),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", 
            config.param_space.l2_leaf_reg.min, 
            config.param_space.l2_leaf_reg.max),
    }


In [9]:
opt_results = hpo.nested_cv_youden_optuna(
    X=X.values,
    y=y.values,
    model_class=hpo.model_class[config.model.name],   # class, not an instance
    param_space_fn=param_space_fn,
    n_splits_outer=config.optimization.k_splits_outer,
    n_repeats_outer=config.optimization.n_repeats_outer,
    n_splits_inner=config.optimization.k_splits_inner,
    n_iter=config.optimization.n_iter,   # Optuna trials per outer fold
    random_state=config.experiment.random_seed,
)

In [10]:
opt_results

{'roc_auc_mean': 0.9724289772727275,
 'roc_auc_std': 0.02043489997771878,
 'youden_mean': 0.8520691287878789,
 'youden_std': 0.06811617456982277,
 'sensitivity_mean': 0.9337357954545455,
 'specificity_mean': 0.9183333333333333,
 'threshold_mean': 0.6352084424142594,
 'threshold_std': 0.122604534242776,
 'folds': [{'fold': 0,
   'roc_auc': 0.9777777777777777,
   'youden': 0.8787878787878789,
   'sensitivity': 0.8787878787878788,
   'specificity': 1.0,
   'threshold': 0.7910987837105941,
   'best_params': {'iterations': 398,
    'depth': 6,
    'learning_rate': 0.025503080154918076,
    'l2_leaf_reg': 1}},
  {'fold': 1,
   'roc_auc': 0.9959595959595959,
   'youden': 0.8666666666666667,
   'sensitivity': 1.0,
   'specificity': 0.8666666666666667,
   'threshold': 0.4922634162487495,
   'best_params': {'iterations': 323,
    'depth': 5,
    'learning_rate': 0.020555556610468626,
    'l2_leaf_reg': 1}},
  {'fold': 2,
   'roc_auc': 0.9541666666666667,
   'youden': 0.8666666666666667,
   'sens

### Calculate Confidence Interval 

In [11]:
opt_ci  = hpo.mean_confidence_interval(opt_results, config)
opt_ci

{'youden': {'mean': 0.8520691287878789,
  'std': 0.06898393152200287,
  'ci_lower': 0.8306911797547911,
  'ci_upper': 0.8734470778209666,
  'n_folds': 40},
 'roc_auc': {'mean': 0.9724289772727275,
  'std': 0.02069522766985295,
  'ci_lower': 0.9660155776216341,
  'ci_upper': 0.9788423769238208,
  'n_folds': 40}}

### Optimization Summary

In [12]:
opt_results_summary = {
    'youden': f'{opt_results["youden_mean"]:.3f} +/ {opt_results["youden_std"]:.3f}',
    'youden ci': f'{opt_ci["youden"]["ci_lower"]:.3f} - {opt_ci["youden"]["ci_upper"]:.3f}', 
    'roc_auc': f'{opt_results["roc_auc_mean"]:.3f} +/ {opt_results["roc_auc_std"]:.3f}',
    'roc_auc ci': f'{opt_ci["roc_auc"]["ci_lower"]:.3f} - {opt_ci["roc_auc"]["ci_upper"]:.3f}', 
    'threshold': f'{opt_results["threshold_mean"]:.3f} +/ {opt_results["threshold_std"]:.3f}',
    'specificity mean': f'{opt_results["specificity_mean"]:.3f}',
    'sensitivity mean': f'{opt_results["sensitivity_mean"]:.3f}',
}

opt_results_df = pd.DataFrame(opt_results_summary, index=['value']).T
opt_results_df

,value
youden,0.852 +/ 0.068
youden ci,0.831 - 0.873
roc_auc,0.972 +/ 0.020
roc_auc ci,0.966 - 0.979
threshold,0.635 +/ 0.123
specificity mean,0.918
sensitivity mean,0.934


 -----

### Train final model on ALL data

In [13]:
catboost_model = CatBoostClassifier(
    verbose=0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=config.experiment.random_seed, 
    thread_count=-1
)

In [14]:
param_space = {
    'iterations': Integer(
        config.param_space.iterations.min, 
        config.param_space.iterations.max),
    'depth': Integer(
        config.param_space.depth.min, 
        config.param_space.depth.max),
    'learning_rate': Real(
        config.param_space.learning_rate.min, 
        config.param_space.learning_rate.max, 
        prior='log-uniform'),  # log-uniform better for LR
    'l2_leaf_reg': Real(
        config.param_space.l2_leaf_reg.min, 
        config.param_space.l2_leaf_reg.max, 
        prior='uniform'),
}

In [15]:
config.final_training

namespace(k_splits_inner=3, n_iter=30)

In [16]:
final_model, best_params = hpo.train_final_model(
    X=X.values, 
    y=y.values, 
    model=catboost_model,
    param_space=param_space,
    n_splits_inner=config.final_training.k_splits_inner,
    n_iter=config.final_training.n_iter, 
    random_state=config.experiment.random_seed, 
    n_jobs=1
)

In [17]:
print(best_params) 

OrderedDict({'depth': 4, 'iterations': 500, 'l2_leaf_reg': 4.930340353636142, 'learning_rate': 0.032559471569963375})


### Final Model Sample Prediction

In [51]:
def test_model(model, threshold):
    test_size = 100
    Xnew = X.iloc[:test_size].values
    ypred, ypredproba = hpo.model_predict(Xnew, model, threshold)
    ynew=y.iloc[:test_size].values

    cm = confusion_matrix(ynew, ypred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    youden_test = (
        sensitivity + specificity - 1
        if not (np.isnan(sensitivity) or np.isnan(specificity))
        else np.nan
    )
    roc_auc = (
        roc_auc_score(ypred, ypredproba)
        if len(np.unique(ynew)) > 1
        else np.nan
    )

    print(cm)
    print('youden: ', youden_test)
    print('roc_auc: ', roc_auc)
    return youden_test, roc_auc

In [52]:
hpo_results_path = config_path /  config.experiment.tag / config.experiment.stage / 'optimization_results.joblib'
optimization_results = joblib.load(hpo_results_path)
threshold_mean = optimization_results['threshold_mean']
threshold_mean

0.6352084424142594

In [64]:
# test_final_model(final_model, final_threshold)
final_model_youden, final_model_roc_auc = test_model(final_model, threshold_mean)

[[28  0]
 [ 0 72]]
youden:  1.0
roc_auc:  1.0


### Unoptimized Model Evaluation

In [65]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

unoptimized_model = CatBoostClassifier(verbose=0)
unoptimized_model.fit(X_train.values, y_train.values)
test_model(unoptimized_model, 0.5)

[[27  1]
 [ 2 70]]
youden:  0.9365079365079365
roc_auc:  1.0


(0.9365079365079365, 1.0)

 -----

### Save artifacts and metrics

#### Final Model

In [74]:
model_version = datetime.now().strftime("%Y%m%d-%H%M")
model_version

'20260312-1051'

In [75]:
final_model_dict = {
    'name': config.model.name,
    'model': final_model,
    'version' : model_version,
    'best_params': best_params,
    'threshold' : threshold_mean,
    'youden' : final_model_youden, 
    'roc_auc' : final_model_roc_auc,
    'n_splits_inner' : config.final_training.k_splits_inner,
    'n_iter' : config.final_training.n_iter, 
}

In [76]:
model_version

'20260312-1051'

In [77]:
joblib.dump(final_model_dict, outputdir / f"final_model_v{model_version}.joblib");


#### Metrics

In [ ]:
joblib.dump(opt_results, outputdir / "optimization_results.joblib");
opt_results_df.to_csv(outputdir / "optimization_results_summary.csv")

### Load Saved Artifacts and Verify

In [80]:
loaded_optimization_results = joblib.load(outputdir / "optimization_results.joblib")
loaded_optimization_results

{'roc_auc_mean': 0.9724289772727275,
 'roc_auc_std': 0.02043489997771878,
 'youden_mean': 0.8520691287878789,
 'youden_std': 0.06811617456982277,
 'sensitivity_mean': 0.9337357954545455,
 'specificity_mean': 0.9183333333333333,
 'threshold_mean': 0.6352084424142594,
 'threshold_std': 0.122604534242776,
 'folds': [{'fold': 0,
   'roc_auc': 0.9777777777777777,
   'youden': 0.8787878787878789,
   'sensitivity': 0.8787878787878788,
   'specificity': 1.0,
   'threshold': 0.7910987837105941,
   'best_params': {'iterations': 398,
    'depth': 6,
    'learning_rate': 0.025503080154918076,
    'l2_leaf_reg': 1}},
  {'fold': 1,
   'roc_auc': 0.9959595959595959,
   'youden': 0.8666666666666667,
   'sensitivity': 1.0,
   'specificity': 0.8666666666666667,
   'threshold': 0.4922634162487495,
   'best_params': {'iterations': 323,
    'depth': 5,
    'learning_rate': 0.020555556610468626,
    'l2_leaf_reg': 1}},
  {'fold': 2,
   'roc_auc': 0.9541666666666667,
   'youden': 0.8666666666666667,
   'sens

In [82]:
loaded_final_model_dict = joblib.load(outputdir / f"final_model_v{model_version}.joblib")
loaded_final_model_dict

{'name': 'catboost',
 'model': <catboost.core.CatBoostClassifier at 0x179fa07dfa0>,
 'version': '20260312-1051',
 'best_params': OrderedDict([('depth', 4),
              ('iterations', 500),
              ('l2_leaf_reg', 4.930340353636142),
              ('learning_rate', 0.032559471569963375)]),
 'threshold': 0.6352084424142594,
 'youden': 1.0,
 'roc_auc': 1.0,
 'n_splits_inner': 3,
 'n_iter': 30}

In [83]:
test_model(
    loaded_final_model_dict["model"], 
    loaded_final_model_dict["threshold"]) 

[[28  0]
 [ 0 72]]
youden:  1.0
roc_auc:  1.0


(1.0, 1.0)

In [84]:
pd.read_csv(outputdir / "optimization_results_summary.csv")

,Unnamed: 0,value
0,youden,0.852 +/ 0.068
1,youden ci,0.831 - 0.873
2,roc_auc,0.972 +/ 0.020
3,roc_auc ci,0.966 - 0.979
4,threshold,0.635 +/ 0.123
5,specificity mean,0.918
6,sensitivity mean,0.934
